In [ ]:
!pip install -q torch transformers accelerate bitsandbytes

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_adi = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_adi)
soru = (
    "İHA'da sinyal kaybı 15 saniyeyi aşarsa hangi prosedür başlatılır? Tek"
    " cümleyle yanıtla."
)


def vram_gb():
  # GPU üzerinde şu an ayrılmış belleği GB cinsinden döndürür
  return torch.cuda.memory_allocated() / (1024**3)


def cevap_ve_sure_olct(model, prompt):
  mesajlar = [{"role": "user", "content": prompt}]
  metin = tokenizer.apply_chat_template(
      mesajlar, tokenize=False, add_generation_prompt=True
  )
  girdi = tokenizer([metin], return_tensors="pt").to(model.device)

  # GPU zamanlaması için senkronizasyon kullanıyoruz
  torch.cuda.synchronize()
  baslangic = time.time()

  with torch.no_grad():
    cikti = model.generate(
        **girdi, max_new_tokens=60, temperature=0.2, do_sample=True
    )

  torch.cuda.synchronize()
  gecen_sure = time.time() - baslangic

  yeni_tokenler = cikti[0][girdi.input_ids.shape[1] :]
  cevap = tokenizer.decode(yeni_tokenler, skip_special_tokens=True)
  return cevap, gecen_sure


# --- 1. DENEY: FP16 (16-bit, sıkıştırılmamış) ---
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_adi, torch_dtype=torch.float16, device_map="auto"
)
fp16_vram = vram_gb()
fp16_cevap, fp16_sure = cevap_ve_sure_olct(model_fp16, soru)

print("=== 16-BIT (FP16) SONUÇLARI ===")
print(f"Kullanılan VRAM : {fp16_vram:.2f} GB")
print(f"Yanıt Süresi   : {fp16_sure:.2f} sn")
print(f"Cevap          : {fp16_cevap}\n")

# Belleği boşaltalım ki 4-bit ölçümü temiz olsun
del model_fp16
torch.cuda.empty_cache()

# --- 2. DENEY: 4-BIT QUANTIZATION (NF4) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # NormalFloat 4-bit: ağırlık dağılımına uygun 4-bit format
    bnb_4bit_compute_dtype=torch.float16,  # Hesaplama sırasında float16 kullan
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_adi, quantization_config=bnb_config, device_map="auto"
)
q4_vram = vram_gb()
q4_cevap, q4_sure = cevap_ve_sure_olct(model_4bit, soru)

print("=== 4-BIT (NF4) SONUÇLARI ===")
print(f"Kullanılan VRAM : {q4_vram:.2f} GB")
print(f"Yanıt Süresi   : {q4_sure:.2f} sn")
print(f"Cevap          : {q4_cevap}\n")

print("=== ÖZET KIYAS ===")
print(f"VRAM Azalması : {fp16_vram - q4_vram:.2f} GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

=== 16-BIT (FP16) SONUÇLARI ===
Kullanılan VRAM : 2.88 GB
Yanıt Süresi   : 2.05 sn
Cevap          : İHA'nın sinyal kaybı 15 saniyeye ulaşana kadar bir prosedür çalışmaz.



Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== 4-BIT (NF4) SONUÇLARI ===
Kullanılan VRAM : 1.14 GB
Yanıt Süresi   : 1.19 sn
Cevap          : İHA (Intelligent Health Assistant) tarafından başlatılan bir prosedür.

=== ÖZET KIYAS ===
VRAM Azalması : 1.74 GB
